
###### 10_embeddings

###### Purpose
The purpose of this notebook is to generate embeddings for Teloc Churn Customer notes using an embedding model and produce dense vector representations that capture the semantic meaning of the text. These embeddings will be used in later notebooks for Vector Search, RAG, and Agentic AI applications.

###### Technologies Used

-  Databricks

-  Databricks Foundation Model APIs

-  Databricks SDK

-  PySpark

-  Delta Lake

-  Unity Catalog

###### Input

Input

-  Sample Telco customer notes

-  Embedding model endpoint: databricks-gte-large-en

-  Unity Catalog table configuration


######  Output

Output

- Customer notes Delta table: NOTES_TABLE

-  Customer-note embeddings Delta table: EMBEDDINGS_TABLE

-  One dense vector for each customer note


######  Architecture

```text

customer_notes
        ↓
Embedding Model
(databricks-gte-large-en)
        ↓
Dense Vector Embeddings
        ↓
Customer Note Embeddings Delta Table

```


###### Section 1 : Call project config notebook

In [0]:
%run ./00_project_config

###### Section 2 : Imports

In [0]:
from databricks.sdk import WorkspaceClient
from pyspark.sql.functions import size, col
from pyspark.sql.types import (
    StructType,
    StructField,
    IntegerType,
    StringType,
    ArrayType,
    DoubleType
)


w = WorkspaceClient()

###### Section 3 : Create customer notes

In [0]:
from pyspark.sql import functions as F

gold_df = spark.table(GOLD_TABLE)

notes_df = (
    gold_df
    .select(
        F.col("customerID").cast("string").alias("customer_id"),

        F.concat_ws(
            " ",
            F.concat(
                F.lit("Customer has a "),
                F.lower(F.col("Contract")),
                F.lit(" contract.")
            ),
            F.concat(
                F.lit("Tenure is "),
                F.col("tenure").cast("string"),
                F.lit(" months.")
            ),
            F.concat(
                F.lit("Monthly charges are $"),
                F.format_number(F.col("MonthlyCharges"), 2),
                F.lit(".")
            ),
            F.concat(
                F.lit("Internet service is "),
                F.col("InternetService"),
                F.lit(".")
            ),
            F.concat(
                F.lit("Technical support status is "),
                F.col("TechSupport"),
                F.lit(".")
            ),
            F.concat(
                F.lit("Online security status is "),
                F.col("OnlineSecurity"),
                F.lit(".")
            ),
            F.concat(
                F.lit("Payment method is "),
                F.col("PaymentMethod"),
                F.lit(".")
            )
        ).alias("note")
    )
)

display(notes_df)

###### Section 4 :  Save Source Notes to Unity Catalog

In [0]:
notes_df.write.format("delta").mode("overwrite").saveAsTable(NOTES_TABLE)

####### Section 5: Validate the table

In [0]:
notes_table_df = spark.table(NOTES_TABLE)

print("Gold rows:", gold_df.count())
print("Notes rows:", notes_table_df.count())
print(
    "Distinct customer IDs:",
    notes_table_df.select("customer_id").distinct().count()
)

notes_table_df.printSchema()
display(notes_table_df.limit(10))

###### Section 5 : Generate Customer-Note Embeddings

In [0]:

# Appropriate for this small learning dataset.
# Use batch inference for large production datasets.

notes = spark.table(NOTES_TABLE).toPandas()

embedding_rows = []

for row in notes.itertuples(index=False):

    if not row.customer_id:
        raise ValueError("Customer ID cannot be empty.")

    if not row.note or not row.note.strip():
        raise ValueError(
            f"Customer {row.customer_id} has an empty note."
        )

    response = w.serving_endpoints.query(
        name=EMBEDDING_MODEL,
        input=[row.note]
    )

    if not response.data or response.data[0].embedding is None:
        raise ValueError(
            f"No embedding returned for customer {row.customer_id}"
        )

    embedding = [
        float(value)
        for value in response.data[0].embedding
    ]

    embedding_rows.append(
        (
            str(row.customer_id),
            row.note,
            embedding
        )
    )

###### Section 6 : Create Embedding DataFrame

In [0]:
embedding_schema = StructType([
    StructField("customer_id", StringType(), False),
    StructField("note", StringType(), False),
    StructField(
        "embedding",
        ArrayType(DoubleType(), containsNull=False),
        False
    )
])

embedding_df = spark.createDataFrame(
    embedding_rows,
    schema=embedding_schema
)

display(embedding_df)

###### Section 7 : Save Embeddings to the Delta table

In [0]:
embedding_df.write.format("delta").mode("overwrite").saveAsTable(EMBEDDINGS_TABLE)

###### Section 8 : Validate Embeddings

In [0]:
saved_embeddings_df = spark.table(EMBEDDINGS_TABLE)

validation_df = saved_embeddings_df.select(
    "customer_id",
    "note",
    size("embedding").alias("embedding_dimension")
)

display(validation_df)

dimensions = [
    row["embedding_dimension"]
    for row in (
        validation_df
        .select("embedding_dimension")
        .distinct()
        .collect()
    )
]

if len(dimensions) != 1:
    raise ValueError(
        f"Inconsistent embedding dimensions found: {dimensions}"
    )

invalid_embedding_count = (
    saved_embeddings_df
    .filter(
        col("embedding").isNull() |
        (size("embedding") == 0)
    )
    .count()
)

assert invalid_embedding_count == 0

print(f"Embedding rows: {saved_embeddings_df.count()}")
print(f"Embedding dimension: {dimensions[0]}")
print("Embedding validation completed successfully.")

###### Notebook Summary

-  Created a sample Telco customer-notes dataset.

-  Stored the source notes in a Unity Catalog Delta table.

-  Generated one dense vector embedding for each customer note using the Databricks GTE embedding model.

-  Created an explicitly typed Spark DataFrame containing customer IDs, source text, and embeddings.

-  Saved the vectors to EMBEDDINGS_TABLE.

-  Validated that every embedding is present and has a consistent dimension.

######  Key Learnings

- Embedding models convert text into dense numerical vectors.

- Similar meanings produce similar vectors.

- Embeddings capture semantics rather than exact word matches.

- Embeddings are stored alongside their source text and identifiers in Delta so that Vector Search can index the numerical vectors while returning the associated business context.

- The same embedding model must be used for stored documents and incoming queries so that both are represented in the same vector space.

###### Concepts Learned

- Embeddings

- Dense Vectors

- Semantic similarity

- Vector Dimensions

- Cosine Similarity

- Shared vector space

###### Notebook Conclusion

- In this notebook, we built an embedding generation pipeline that converts customer notes text into dense vector embeddings using the Databricks GTE embedding model and stores them in a Delta table.

- This enables semantic representation of text, allowing AI systems to understand meaning rather than relying on exact keyword matching.

- This embedding table will be used in the next notebook to create a Vector Search index for semantic retrieval of relevant customer notes.

- Below questions are answered:

    1. What is an embedding? --> Numerical representation of text meaning.
    2. How do we generate embeddings? --> Pass text to an embedding model and receive a vector representing the meaning of that text.
    3. Where are embeddings stored? -->Delta table.
    4. Why do we need embeddings before Vector Search? -->Vector Search works on vectors, not text.
    5. Why is the same embedding model used for documents and questions? --->So both live in the same vector space and similarity search works.



###### Next Notebook

11_vector_search

- Create a Databricks Vector Search endpoint and Delta Sync index over the customer-note embeddings table. The index will support semantic retrieval for the RAG and Agentic AI workflows.

In [0]:
EMBEDDINGS_TABLE

In [0]:
%sql
select * from dbw_agentic_ai_dev.telco_ai.customer_note_embeddings